# Hung matched Development — fixed shards 0002 and 0003
40 jobs, disjoint from the local queue. The predecessor archive and current quota must pass before execution. No Final evaluation or independent acceptance claim.

In [ ]:
import hashlib
import importlib.util
import json
import os
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import torch

base = Path("/kaggle/working/hung-matched-group02-v1")
assert not base.exists(), "Preserve existing output; do not run this version twice"
payloads = list(Path("/kaggle/input").rglob("provider-payload.zip.bin"))
assert len(payloads) == 1, "Attach exact private payload dataset"
payload = payloads[0]
helper = payload.parent / "hung_provider.py"
assert (
    hashlib.sha256(helper.read_bytes()).hexdigest()
    == "cadcd80d671d3b49a317312f125735362ffba3068247f1714a0ae56c6be83424"
)
assert (
    hashlib.sha256(payload.read_bytes()).hexdigest()
    == "23a6d420523d990d07ff07d814ed642a851925283a102a3ae806f3b666e1d6f4"
)
spec = importlib.util.spec_from_file_location("hung_provider", helper)
provider = importlib.util.module_from_spec(spec)
spec.loader.exec_module(provider)
repo, plan_path = provider.materialize(
    payload, "23a6d420523d990d07ff07d814ed642a851925283a102a3ae806f3b666e1d6f4", base
)
os.chdir(repo)
sys.path.insert(0, str(repo))

assert torch.cuda.is_available(), "CUDA GPU required"
assert shutil.disk_usage(base).free >= 8 * 1024**3, "8 GiB free required"
environment = {
    "torch": torch.__version__,
    "python": sys.version,
    "cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0),
    "memory_bytes": torch.cuda.get_device_properties(0).total_memory,
    "disk_free_bytes": shutil.disk_usage(base).free,
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "quota_reference_at_preparation": "1406.8571645999998 GPU minutes available, owner UI at 2026-10-26T20:51:14.219032+00:00; recheck before later allocation",
    "code_sha": "44b8532959d76fb20691881539f991ca44983727",
    "shards": ["shard_0002", "shard_0003"],
    "scope": "Development execution authorized by user; independent acceptance and Final remain pending",
}
provider.save(base / "environment.json", environment)
print(json.dumps(environment))
assert torch.__version__ == "2.11.0+cu128", "Provider image changed; revalidate before execution"

In [ ]:
import math
import zipfile
from datetime import datetime, timezone

from kagglehub.clients import build_kaggle_client
from kagglesdk.kernels.types.kernels_api_service import ApiGetAcceleratorQuotaStatisticsRequest

QUOTA = {
    "available_gpu_minutes": 1406.8571645999998,
    "observed_at_utc": "2026-10-26T20:51:14.219032+00:00",
    "raw_gpu_quota": {
        "observed_at_utc": "2026-10-26T20:51:14.219032+00:00",
        "pay_to_scale_enabled": False,
        "reserved_seconds": 0.0,
        "total_seconds": 108000.0,
        "used_seconds": 23588.570124,
    },
    "scope": "Read-only quota succeeded; storage-status endpoint separately returned403",
    "source": "Authenticated official SDK GetAcceleratorQuotaStatistics in CPU notebook version362621233",
}
QUOTA_INITIAL = QUOTA
with build_kaggle_client() as quota_client:
    gpu_quota = quota_client.kernels.kernels_api_client.get_accelerator_quota_statistics(
        ApiGetAcceleratorQuotaStatisticsRequest()
    ).gpu_quota
    assert gpu_quota is not None
    assert all(
        v is not None
        for v in (gpu_quota.total_time_allowed, gpu_quota.time_used, gpu_quota.time_reserved)
    )
    assert not gpu_quota.is_pay_to_scale_enabled, "No paid GPU scaling authorized"
    QUOTA = {
        "observed_at_utc": datetime.now(timezone.utc).isoformat(),
        "available_gpu_minutes": (
            gpu_quota.total_time_allowed - gpu_quota.time_used - gpu_quota.time_reserved
        ).total_seconds()
        / 60,
        "used_seconds": gpu_quota.time_used.total_seconds(),
        "reserved_seconds": gpu_quota.time_reserved.total_seconds(),
        "total_seconds": gpu_quota.total_time_allowed.total_seconds(),
        "source": "Authenticated official SDK quota at execution",
    }
assert 1018.35 <= QUOTA["available_gpu_minutes"] - 60, (
    "Measured remaining-matrix quota feasibility fails"
)
observed = datetime.fromisoformat(QUOTA["observed_at_utc"])
assert 0 <= (datetime.now(timezone.utc) - observed).total_seconds() <= 1800, (
    "Refresh owner quota observation"
)
PREDECESSOR = {
    "provider_wall_seconds": 1810.4,
    "receipt": {
        "artifact_bytes": 65931034,
        "artifact_sha256": "29e3db51106402942959f80762e4c27995ad4ec9f2253fd96cc8a4681d3b2dcd",
        "crc": "passed",
        "ended_at_utc": "2026-10-26T20:32:16.508290+00:00",
        "exit_code": 0,
        "job_ids": [
            "block28_b14s11e1025",
            "block28_b14s11e1153",
            "block28_b14s11e1405",
            "block28_b14s11e1693",
            "block28_b14s11e1987",
            "block28_b14s11e2152",
            "block28_b14s11e2613",
            "block28_b14s11e2644",
            "block28_b14s19e1025",
            "block28_b14s19e1153",
            "block28_b14s19e1405",
            "block28_b14s19e1693",
            "block28_b14s19e1987",
            "block28_b14s19e2152",
            "block28_b14s19e2613",
            "block28_b14s19e2644",
            "block28_b14s7e1025",
            "block28_b14s7e1153",
            "block28_b14s7e1405",
            "block28_b14s7e1693",
        ],
        "local_verify_exit_code": 0,
        "peer_acceptance": "pending",
        "plan_sha256": "80a1666cf2083594c9a2bc0b0a4dc3e30118eda43d07187cd5b6d8b51f90e0bb",
        "remote_readback": "pending",
        "shard_id": "shard_0001",
        "shards_sha256": "e2c6cb6f15a7473c6cdfd7912fefa838d63a770cf352dff93c2cccc3ae86a15c",
        "wall_seconds": 1298.752902966,
    },
    "scope": "Execution, CPU verification and runtime-local CRC; remote output readback remains pending",
    "terminal_ui_status": "1810.4 second run - successful",
    "version_url": "https://www.kaggle.com/code/hungluongduc/hung-matched-shard0001-44b8532-20261027?scriptVersionId=362612528",
    "visible_log_sha256": "f1b094162a206f11d36b79767a0e2f228b3f0ccb7f039d39e80930393012e63f",
}
matches = [p for p in Path("/kaggle/input").rglob("artifacts.zip") if p.parent.name == "shard_0001"]
assert len(matches) == 1, "Attach completed private predecessor version output"
archive = matches[0]
readback_started = time.perf_counter()
assert archive.stat().st_size == PREDECESSOR["receipt"]["artifact_bytes"]
assert provider.sha256(archive) == PREDECESSOR["receipt"]["artifact_sha256"]
with zipfile.ZipFile(archive) as stream:
    members = provider.safe_members(stream)
    identity = json.loads(stream.read("dispatch_identity.json"))
    assert identity["plan_sha256"] == PREDECESSOR["receipt"]["plan_sha256"]
    measured = {}
    peak_allocated = 0
    peak_reserved = 0
    updates = {}
    for name in members:
        if name.startswith("jobs/") and name.endswith("/dispatch_measurement.json"):
            job = name.split("/")[1]
            measured[job] = measured.get(job, 0) + json.loads(stream.read(name))["wall_seconds"]
        if name.startswith("jobs/") and name.endswith("/run_manifest.json"):
            manifest = json.loads(stream.read(name))
            job = name.split("/")[1]
            device = manifest.get("device", {})
            if isinstance(device, dict):
                peak_allocated = max(peak_allocated, device.get("peak_allocated_bytes", 0))
                peak_reserved = max(peak_reserved, device.get("peak_reserved_bytes", 0))
            costs = manifest.get("runtime_costs", {})
            count = sum(c.get("update_steps", 0) for c in costs.get("candidate", []))
            count += costs.get("audit", {}).get("update_steps", 0) + manifest.get(
                "proxy_costs", {}
            ).get("update_steps", 0)
            updates[job] = max(updates.get(job, 0), count)
    assert set(measured) == set(PREDECESSOR["receipt"]["job_ids"])
    job_times = sorted(measured.values())
    p95 = job_times[math.ceil(0.95 * len(job_times)) - 1]
    uncompressed_bytes = sum(info.file_size for info in stream.infolist())
readback = {
    "predecessor_version": PREDECESSOR["version_url"],
    "artifact_sha256": provider.sha256(archive),
    "crc": "passed",
    "jobs": len(measured),
    "p95_job_subprocess_seconds": p95,
    "p95_scope": "Per-job subprocess sums; excludes orchestration, independent verification and archive overhead",
    "measured_whole_provider_seconds": PREDECESSOR["provider_wall_seconds"],
    "peak_cuda_allocated_bytes": peak_allocated,
    "peak_cuda_reserved_bytes": peak_reserved,
    "replay_update_steps": sum(updates.values()),
    "update_scope": "Cumulative per-job maxima across continuation stages; includes candidate/audit/proxy updates, excludes reused Source/Target training",
    "artifact_uncompressed_bytes": uncompressed_bytes,
    "hash_crc_readback_seconds": time.perf_counter() - readback_started,
    "projected_remaining_gpu_minutes_with_1_25_margin": 1018.35,
    "quota_observation": QUOTA,
    "acceptance": "Technical durability only; independent peer acceptance pending",
}
provider.save(base / "predecessor-readback.json", readback)
print(json.dumps(readback))

In [ ]:
# Bounded CPU/CUDA portability check on two permitted initial Development examples.
import json
from copy import deepcopy
from pathlib import Path

import torch

from experiments.hung_sprint2 import dispatch_matched as dispatch
from experiments.hung_sprint2.cli import _examples, _tensors
from experiments.hung_sprint2.dispatch_state import atomic_create
from experiments.hung_sprint2.model import ForecastGRU, ModelConfig
from experiments.hung_sprint2.retention import retention_loss

torch.set_num_threads(1)
ATOL, RTOL = 1e-5, 1e-4
root = Path.cwd()
plan_path = root / ".cache/provider/plan.json"
plan = dispatch.read_verified_json(plan_path, dispatch.sha256_file(plan_path))
job = next(row for row in plan["jobs"] if row["id"] == "online_best_b14s7e1153")
cell = next(row for row in plan["cells"] if row["id"] == job["cell"])
config_pin = next(row for row in plan["configs"] if row["id"] == job["config"])
config = dispatch.read_verified_json(Path(config_pin["path"]), config_pin["sha256"])
input_run = Path(plan["input_base"]) / "urbanev-SZH-14d"
manifest = dispatch._load_run(input_run, cell["input_manifest_sha256"])
rows, count = _examples(input_run, manifest, "development", cell["entity_id"], limit=2)
assert rows and torch.cuda.is_available()
x, c, y = _tensors(rows)
states = {}
for name in ("source", "target"):
    dispatch.verify_file(Path(job[name]["path"]), job[name]["sha256"])
    states[name] = torch.load(job[name]["path"], map_location="cpu", weights_only=True)


def evaluate(device):
    target = ForecastGRU(ModelConfig(**states["target"]["architecture"])).to(device).train()
    source = ForecastGRU(ModelConfig(**states["source"]["architecture"])).to(device).eval()
    target.load_state_dict(states["target"]["model_state"])
    source.load_state_dict(states["source"]["model_state"])
    source.requires_grad_(False)
    optimizer = torch.optim.Adam(target.parameters(), lr=config["target"]["learning_rate"])
    optimizer.load_state_dict(deepcopy(states["target"]["optimizer_state"]))
    prediction, latent = target.forecast_and_latent(x.to(device), c.to(device))
    with torch.no_grad():
        anchor = source.encode(x.to(device))
    forecast = (prediction - y.to(device)).abs().mean()
    retain = retention_loss(
        latent,
        anchor,
        states["source"]["latent_scale"].to(device),
        epsilon=config["target"]["epsilon"],
    )
    loss = forecast + config["target"]["retention_weight"] * retain
    loss.backward()
    result = {
        "forecast": prediction.detach().cpu(),
        "latent": latent.detach().cpu(),
        "forecast_loss": forecast.detach().cpu(),
        "retention_loss": retain.detach().cpu(),
        **{
            "gradient/" + name: parameter.grad.detach().cpu().clone()
            for name, parameter in target.named_parameters()
        },
    }
    optimizer.step()
    result.update(
        {
            "updated/" + name: parameter.detach().cpu().clone()
            for name, parameter in target.named_parameters()
        }
    )
    return result


started = time.perf_counter()
cpu, gpu = evaluate("cpu"), evaluate("cuda")
checks = {
    key: {
        "max_absolute_difference": float((cpu[key] - gpu[key]).abs().max()),
        "within_tolerance": bool(torch.allclose(cpu[key], gpu[key], atol=ATOL, rtol=RTOL)),
    }
    for key in cpu
}
passed = all(row["within_tolerance"] for row in checks.values())
report = {
    "schema": "hung-local-cuda-bounded-parity-v1",
    "passed": passed,
    "atol": ATOL,
    "rtol": RTOL,
    "job_id": job["id"],
    "plan_sha256": dispatch.sha256_file(plan_path),
    "code": dispatch._code_identity(),
    "input_manifest_sha256": cell["input_manifest_sha256"],
    "source": job["source"],
    "target": job["target"],
    "permitted_initial_examples_used": len(rows),
    "permitted_initial_examples_available": count,
    "elapsed_seconds": time.perf_counter() - started,
    "device": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "checks": checks,
    "scope": "Initial Development forward/loss/gradient/one-Adam-update portability only; no saved model update or research result",
}
atomic_create(root.parent / "provider-parity.json", report)
print(
    json.dumps(
        {"passed": passed, "checks": len(checks), "elapsed_seconds": report["elapsed_seconds"]}
    )
)
assert passed, "Provider parity failed; do not execute shard"

In [ ]:
assert passed, "Frozen CPU/CUDA parity prerequisite"
receipts = []
group_start = time.perf_counter()
for shard_id in ["shard_0002", "shard_0003"]:
    assert time.perf_counter() - group_start < 6 * 3600, (
        "Stop before another shard after bounded session budget"
    )
    receipt = provider.execute_shard(repo, shard_id, base / shard_id)
    receipts.append(receipt)
    print(json.dumps(receipt), flush=True)
complete = {
    "shards": ["shard_0002", "shard_0003"],
    "jobs": sum(len(r["job_ids"]) for r in receipts),
    "receipts": receipts,
    "wall_seconds": time.perf_counter() - group_start,
    "allocation_queue_sha256": "12f5bc9ee402a806faedc89574a4e00c427c384335a3868668adf8ce7f93a0f2",
    "predecessor_sha256": PREDECESSOR["receipt"]["artifact_sha256"],
    "remote_readback": "Pending next provider readback; only a completed version preserves these outputs",
    "scope": "Development execution; no Final lock or peer-acceptance claim",
}
provider.save(base / "group-complete.json", complete)
print(json.dumps(complete), flush=True)